# llm-t4-lab на Kaggle

Настройки ноутбука: **Accelerator — GPU T4 x2**, **Internet — on**.

Все шаги запускаются как отдельные процессы (`!python -m ...`), поэтому после установки пакетов перезапуск ядра не нужен. Разделы независимы: если сессия оборвалась, можно продолжить с нужного места. Результаты копятся в `results/`, в конце собираются в `results.zip`.

In [ ]:
!git clone https://github.com/crptptn/llm-t4-lab.git
%cd llm-t4-lab
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"

In [ ]:
SMALL = "Qwen/Qwen2.5-1.5B-Instruct"  # бенчмарк и база для QLoRA
BIG = "Qwen/Qwen2.5-7B-Instruct-AWQ"  # генерация данных, судья RAG, tool calling

## 1. Бенчмарк vLLM

Три прогона: обычные запросы, затем общий системный промпт с prefix caching и без него.

In [ ]:
!python -m lab.server start {SMALL} -- --enable-prefix-caching
!python -m lab.bench --model {SMALL} --scenario unique --tag unique
!python -m lab.bench --model {SMALL} --scenario shared-prefix --tag prefix_on
!python -m lab.server stop

In [ ]:
!python -m lab.server start {SMALL} -- --no-enable-prefix-caching
!python -m lab.bench --model {SMALL} --scenario shared-prefix --tag prefix_off
!python -m lab.server stop

## 2. Квантизация: FP16 / AWQ / GPTQ на GSM8K

In [ ]:
!python -m lab.quant --n 300

## 3. RAG: поиск

BM25, e5, гибрид и reranker на SberQuAD. Сервер не нужен, модели поиска работают на GPU напрямую.

In [ ]:
!python -m lab.rag_eval retrieval --n-queries 1000

## 4. Модель 7B: синтетические обращения, tool calling, faithfulness

Генерация датасета — самый долгий шаг раздела.

In [ ]:
!python -m lab.server start {BIG} -- --enable-auto-tool-choice --tool-call-parser hermes
!python -m lab.appeals.generate --model {BIG}
!python -m lab.tools_eval all --model {BIG}
!python -m lab.rag_eval faithfulness --model {BIG} --n 150
!python -m lab.server stop

## 5. QLoRA-дообучение классификатора

In [ ]:
!python -m lab.appeals.train

## 6. Оценка: база против LoRA-адаптера в vLLM

Если vLLM не поднимет адаптер, можно обучить с `--save-merged` и раздать склеенную модель из `outputs/appeals-lora-merged`.

In [ ]:
!python -m lab.server start {SMALL} -- --enable-lora --lora-modules appeals=outputs/appeals-lora --max-lora-rank 16
!python -m lab.appeals.evaluate --base-model {SMALL}
!python -m lab.server stop

## 7. Отчёт

`results.zip` появится во вкладке Output. Его содержимое кладётся в репозиторий поверх старого.

In [ ]:
!python -m lab.report
!zip -qr /kaggle/working/results.zip results RESULTS.md data/appeals
!head -40 RESULTS.md

In [ ]:
# если что-то упало при старте сервера
!python -m lab.server log